# Mars crater correction visualisation

This notebook compares the supplied 2012 `regularhd-large` images and labels with the corrected dataset. The image archives remain external to the repository. Change `TILE_STEM` or use the interactive browser to inspect different examples.

## 1. Import libraries and configure paths

The notebook expects to run from the repository or any subdirectory beneath it. The source and corrected datasets are deliberately kept outside the repository.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from PIL import Image, ImageChops


def find_repository_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "tools" / "dataset").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    fallback = Path("/home/dcortereal/label-free-mars-crater-detection")
    if (fallback / "tools" / "dataset").is_dir():
        return fallback
    raise FileNotFoundError("Could not locate the repository root")


REPO_ROOT = find_repository_root()
NOTEBOOK_DIR = REPO_ROOT / "notebooks"
NOTEBOOK_PATH = NOTEBOOK_DIR / "correction_visualization.ipynb"
SOURCE_ROOT = Path("/home/dcortereal/datasets/regularhd-large")
CORRECTED_ROOT = Path("/home/dcortereal/datasets/regularhd-large-corrected-2012")
SOURCE_LABELS_PATH = SOURCE_ROOT / "labels.json"
CORRECTED_LABELS_PATH = CORRECTED_ROOT / "labels.json"
TILE_STEM = "lat_-52.0_long_154.0"

assert NOTEBOOK_DIR.name == "notebooks"
assert NOTEBOOK_PATH.parent == NOTEBOOK_DIR

print(f"Repository: {REPO_ROOT}")
print(f"Notebook: {NOTEBOOK_PATH}")
print(f"Source dataset exists: {SOURCE_ROOT.exists()}")
print(f"Corrected dataset exists: {CORRECTED_ROOT.exists()}")

## 2. Load images and labels

Labels use the project format `[center_x, center_y, radius_x, radius_y]`. The loader keeps that format and pairs each source tile with its corrected counterpart.

In [ ]:
def load_labels(path: Path) -> dict[str, list[list[float]]]:
    return json.loads(path.read_text(encoding="utf-8"))


def discover_stems(root: Path, suffix: str) -> list[str]:
    return sorted(path.name.removesuffix(suffix) for path in root.glob(f"*{suffix}"))


def load_example(stem: str) -> dict:
    original_path = SOURCE_ROOT / f"{stem}_original.png"
    corrected_path = CORRECTED_ROOT / f"{stem}_corrected.png"
    if not original_path.exists():
        raise FileNotFoundError(original_path)
    if not corrected_path.exists():
        raise FileNotFoundError(corrected_path)
    return {
        "stem": stem,
        "original_path": original_path,
        "corrected_path": corrected_path,
        "original_image": Image.open(original_path).convert("RGB"),
        "corrected_image": Image.open(corrected_path).convert("RGB"),
        "original_labels": original_labels.get(stem, []),
        "corrected_labels": corrected_labels.get(stem, []),
    }


original_labels = load_labels(SOURCE_LABELS_PATH)
corrected_labels = load_labels(CORRECTED_LABELS_PATH)
original_stems = set(discover_stems(SOURCE_ROOT, "_original.png"))
corrected_stems = set(discover_stems(CORRECTED_ROOT, "_corrected.png"))
common_stems = sorted(original_stems & corrected_stems)

print(f"Original image tiles: {len(original_stems):,}")
print(f"Corrected image tiles: {len(corrected_stems):,}")
print(f"Paired examples: {len(common_stems):,}")
print(f"Original labels: {sum(map(len, original_labels.values())):,}")
print(f"Corrected labels: {sum(map(len, corrected_labels.values())):,}")
print(f"First examples: {common_stems[:5]}")

## 3. Visualize original images with labels

The original image is shown with the labels from the supplied `regularhd-large/labels.json`. Ellipse centres outside the canvas are retained when the ellipse intersects the image.

In [ ]:
def add_label_overlays(ax, labels: list[list[float]], color: str, linewidth: float = 1.2) -> None:
    for center_x, center_y, radius_x, radius_y in labels:
        ax.add_patch(
            Ellipse(
                (center_x, center_y),
                width=2 * radius_x,
                height=2 * radius_y,
                fill=False,
                edgecolor=color,
                linewidth=linewidth,
            )
        )
        ax.plot(center_x, center_y, marker="+", color=color, markersize=4, markeredgewidth=0.8)


def show_image_and_labels(
    image: Image.Image,
    labels: list[list[float]],
    title: str,
    color: str,
) -> None:
    width, height = image.size
    figure, axes = plt.subplots(1, 2, figsize=(16, 7))
    axes[0].imshow(image, origin="upper")
    axes[0].set_title(f"{title} without labels")
    axes[1].imshow(image, origin="upper")
    add_label_overlays(axes[1], labels, color)
    axes[1].set_title(f"{title} with labels ({len(labels):,})")
    for axis in axes:
        axis.set_xlim(0, width)
        axis.set_ylim(height, 0)
        axis.set_aspect("equal")
        axis.set_xlabel("x (pixels)")
        axis.set_ylabel("y (pixels)")
    plt.tight_layout()
    plt.show()


example = load_example(TILE_STEM)
show_image_and_labels(
    example["original_image"],
    example["original_labels"],
    f"Original: {TILE_STEM}",
    "lime",
)

## 4. Visualize corrected images with labels

The corrected image and `labels.json` are plotted with the same overlay convention. This makes horizontal compression, remosaicking, edge labels, and black coverage gaps easy to inspect.

In [ ]:
show_image_and_labels(
    example["corrected_image"],
    example["corrected_labels"],
    f"Corrected: {TILE_STEM}",
    "red",
)

## 5. Compare before-and-after corrections

The first two panels compare image and label geometry. The third panel is a pixel difference for visual reference only; because the correction remosaics and rescales the image, it is not a pixel-registered error map.

In [ ]:
def show_comparison(example: dict) -> None:
    difference = ImageChops.difference(example["original_image"], example["corrected_image"])
    figure, axes = plt.subplots(1, 3, figsize=(20, 7))
    panels = [
        (example["original_image"], example["original_labels"], "Original", "lime"),
        (example["corrected_image"], example["corrected_labels"], "Corrected", "red"),
        (difference, [], "Pixel difference (not registered)", None),
    ]
    for axis, (image, labels, title, color) in zip(axes, panels):
        axis.imshow(image, origin="upper")
        if color is not None:
            add_label_overlays(axis, labels, color)
        axis.set_title(f"{title}\\n{len(labels):,} labels" if labels else title)
        axis.set_xlim(0, image.width)
        axis.set_ylim(image.height, 0)
        axis.set_aspect("equal")
        axis.axis("off")
    figure.suptitle(example["stem"])
    plt.tight_layout()
    plt.show()


show_comparison(example)

## 6. Browse multiple examples interactively

Use the slider to browse paired source/corrected tiles without changing the notebook code. The slider index follows the sorted tile-stem list.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    def browse_example(index: int = 0) -> None:
        selected_stem = common_stems[index]
        show_comparison(load_example(selected_stem))

    index_slider = widgets.IntSlider(
        value=0,
        min=0,
        max=max(0, len(common_stems) - 1),
        step=1,
        description="Tile index",
        continuous_update=False,
        readout=True,
    )
    display(widgets.interactive(browse_example, index=index_slider))
except ImportError:
    print("Install ipywidgets to enable the interactive browser.")
    print("You can still change TILE_STEM and rerun the static comparison cells.")

## 7. Validate label and image alignment

These checks distinguish expected edge labels from invalid records. A label centre may be outside the canvas when its ellipse still intersects the image.

In [ ]:
import math


def ellipse_intersects_canvas(label: list[float], size: tuple[int, int]) -> bool:
    center_x, center_y, radius_x, radius_y = label
    width, height = size
    return not (
        center_x + radius_x < 0
        or center_x - radius_x > width
        or center_y + radius_y < 0
        or center_y - radius_y > height
    )


def validate_labels(labels: dict[str, list[list[float]]], image_stems: set[str], image_size: tuple[int, int]) -> dict:
    invalid_records = []
    invisible_records = []
    for stem, tile_labels in labels.items():
        for label_index, label in enumerate(tile_labels):
            try:
                values = [float(value) for value in label]
                valid_shape = len(values) == 4
                valid_values = all(math.isfinite(value) for value in values)
                valid_radii = values[2] > 0 and values[3] > 0
            except (TypeError, ValueError):
                valid_shape = valid_values = valid_radii = False
                values = []
            if not (valid_shape and valid_values and valid_radii):
                invalid_records.append((stem, label_index, label))
            elif not ellipse_intersects_canvas(values, image_size):
                invisible_records.append((stem, label_index))
    return {
        "label_keys_without_images": sorted(set(labels) - image_stems),
        "invalid_records": invalid_records,
        "fully_outside_canvas_records": invisible_records,
        "label_count": sum(len(tile_labels) for tile_labels in labels.values()),
    }


original_validation = validate_labels(original_labels, original_stems, (512, 512))
corrected_validation = validate_labels(corrected_labels, corrected_stems, (512, 512))
missing_original_labels = sorted(original_stems - set(original_labels))
missing_corrected_labels = sorted(corrected_stems - set(corrected_labels))

sample_stems = common_stems[:200]
dimension_mismatches = []
for stem in sample_stems:
    with Image.open(SOURCE_ROOT / f"{stem}_original.png") as original_image:
        original_size = original_image.size
    with Image.open(CORRECTED_ROOT / f"{stem}_corrected.png") as corrected_image:
        corrected_size = corrected_image.size
    if original_size != corrected_size:
        dimension_mismatches.append((stem, original_size, corrected_size))

validation_summary = {
    "missing_original_labels": missing_original_labels,
    "missing_corrected_labels": missing_corrected_labels,
    "original": {
        "label_count": original_validation["label_count"],
        "label_keys_without_images": original_validation["label_keys_without_images"],
        "invalid_record_count": len(original_validation["invalid_records"]),
        "fully_outside_record_count": len(original_validation["fully_outside_canvas_records"]),
    },
    "corrected": {
        "label_count": corrected_validation["label_count"],
        "label_keys_without_images": corrected_validation["label_keys_without_images"],
        "invalid_record_count": len(corrected_validation["invalid_records"]),
        "fully_outside_record_count": len(corrected_validation["fully_outside_canvas_records"]),
    },
    "dimension_mismatches_in_sample": dimension_mismatches,
}

from pprint import pprint
pprint(validation_summary)
print(f"Checked dimensions for {len(sample_stems):,} paired examples.")